# 07 — Image Supervised Fine-Tuning (SFT) for a Vision–Language Model (VLM)

## What changes when an image is part of the input?

**Supervised Fine-Tuning (SFT)** learns demonstrated responses. A **Vision–Language Model (VLM)** consumes visual information along with text. **Multimodal** means combining more than one kind of input, such as pixels and language.

An **image patch** is a small region used to represent an image. A **vision encoder** transforms patches into learned feature vectors. A **projector** maps feature dimensions or representations; Qwen's **merger** also combines spatial features before presenting them to the language model. A **visual placeholder** marks positions in the language sequence where image-derived representations belong. It is not itself the image.

A **processor** coordinates image preprocessing and text tokenization according to the checkpoint. A **tokenizer** converts text to integer vocabulary IDs.

**What problem does this solve?** Image SFT teaches responses conditioned on visual evidence, such as reading a status panel. Text-only examples cannot directly teach the same pixel-to-answer association. A visually plausible answer still needs external correctness checks.


## NovaBot: image, question, and target answer

**Fictional annotation:** imagine a local image of NovaBot's panel with a red indicator.

~~~json
{"image":"panels/novabot-red.png",
 "messages":[
   {"role":"user","content":[
     {"type":"image"},
     {"type":"text","text":"What color is NovaBot's indicator?"}
   ]},
   {"role":"assistant","content":"The indicator is red."}
 ]}
~~~

The image path is resolved against an image root, decoded, and passed to the processor. The image content block must correspond to an actual image. The target answer specifies what the model should learn to say; it is not an observed result of training.

**Hand-constructed geometry:** a 32×32 image split into 8×8 spatial patches has 4×4=16 patches. If each 2×2 group is merged, it yields 4 language-side visual positions. This illustrates the tiny fixture's spatial geometry; actual checkpoints can resize images and use temporal grouping, so inspect the processor outputs rather than assuming that every image yields four tokens.


## How image-conditioned answer loss works

Let $I$ denote the processed image, $x$ the text prompt, $y_t$ reference answer token $t$, $y_{<t}$ earlier reference answer tokens, and $\theta$ the selected trainable parameters. The objective is

$$L=-\frac{1}{N}\sum_{t\in A}\log p_\theta(y_t\mid I,x,y_{<t}),$$

where $A$ selects assistant target positions, $N=|A|$, and $\log$ is the natural logarithm. This is answer-token cross-entropy conditioned on both pixels and text.

**Hand calculation:** assigning the target word “red” probability 0.8 contributes about 0.223 to its negative log-likelihood. The same token-loss intuition applies as in text SFT. It is not a pixel reconstruction objective.

Prompt positions, visual placeholder positions and padding have ignored labels (-100). They can still provide context. Masking visual targets does not prevent gradients from flowing through trainable image-processing components that influenced the answer.


## Connection to this notebook

open_image() reads local pixels. The processor produces input_ids, attention_mask, pixel_values and image_grid_thw. The last field describes temporal/height/width patch geometry; pixel_values contains processed patches, not text token IDs.

collate_text() encodes full conversations and prompt prefixes with the same image, checks prefix alignment, and masks everything except the final assistant answer. This lesson accepts one assistant answer per image sample. MAX_IMAGE_SEQUENCE rejects over-budget samples rather than cutting through expanded visual tokens.

**Low-Rank Adaptation (LoRA)** trains small added language matrices; **Quantized Low-Rank Adaptation (QLoRA)** also compresses the frozen base. modules_to_save includes the merger so its updates survive adapter saving. The main vision encoder remains frozen in this lesson's strategy. Frozen means no weight updates, not that its features are unused.

The executable fixtures are generated color images, not the fictional NovaBot file above. Reload verification supplies the image again; generating from the question alone would not test the same behavior.


## Common confusions and quick check

One image placeholder need not become one language position. Truncating visual tokens is not equivalent to safely resizing an image.

1. Does setting visual-position labels to -100 mean the model cannot learn from the image?
2. Can an adapter omit an updated merger and still reproduce the same model?

<details>
<summary>Answers</summary>

1. No. The answer loss can backpropagate through trainable visual-to-language components. The mask only excludes visual positions as prediction targets.
2. Not reliably. Save every trainable component needed to reconstruct the trained behavior, including an updated merger.

</details>


## Before running the experiment

**Learning goals:** decode local images, inspect patches and visual placeholders, construct safe labels, train language adapters plus the merger, and reload for image-conditioned generation.

Run in a fresh kernel. All weights and files are local. Tiny random checkpoints demonstrate mechanics, not model quality. [Course index](README.md) · [Flow diagrams](../docs/EXECUTION_AND_DATA_FLOW.md)

[Terminology reference](../docs/GLOSSARY.md) · [Compare training methods](../docs/TRAINING_METHODS.md)


## Experiment parameters


In [ ]:
LESSON = "07"
# Parameters: change these before running the notebook from top to bottom.
import csv
import json
import math
import os
import random
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForImageTextToText, AutoProcessor

from finetunelab.education import (
    inspect_local_checkpoint,
    make_tiny_checkpoint,
    project_root,
    token_table,
)
from finetunelab.tuning import parameter_report

MODE = os.environ.get("FTLAB_NOTEBOOK_MODE", "tiny_cpu")
LOCAL_MODEL_PATH = Path(os.environ.get("FTLAB_LOCAL_MODEL", "models/Qwen3.5-2B"))
LOCAL_TEACHER_PATH = Path(os.environ.get("FTLAB_LOCAL_TEACHER", "models/Qwen3.5-4B"))
ROOT = project_root()
DATA_ROOT = Path(os.environ.get("FTLAB_LESSON_DATA", str(ROOT / "examples/education")))
OUTPUT_ROOT = Path(os.environ.get("FTLAB_NOTEBOOK_OUTPUT", str(ROOT / "outputs/notebooks")))
OUTPUT = OUTPUT_ROOT / LESSON
OUTPUT.mkdir(parents=True, exist_ok=True)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)
assert MODE in {"tiny_cpu", "local_pretrained"}
DEVICE = torch.device("cpu" if MODE == "tiny_cpu" else "cuda")
if MODE == "local_pretrained" and not torch.cuda.is_available():
    raise RuntimeError(
        "The local_pretrained teaching profile requires a CUDA PyTorch installation."
    )
DTYPE = torch.float32 if MODE == "tiny_cpu" else torch.bfloat16
MODEL_PATH = (
    make_tiny_checkpoint(OUTPUT / "initial", seed=SEED)
    if MODE == "tiny_cpu"
    else LOCAL_MODEL_PATH.expanduser().resolve()
)
checkpoint_info = inspect_local_checkpoint(MODEL_PATH)
print({"mode": MODE, "device": str(DEVICE), "checkpoint": str(MODEL_PATH)})
print(checkpoint_info["files"])

## Load model and processor


In [ ]:
# A checkpoint includes both weights and the preprocessing contract.
processor = AutoProcessor.from_pretrained(MODEL_PATH, local_files_only=True)
tokenizer = processor.tokenizer
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
load_kwargs = {
    "local_files_only": True,
    "dtype": DTYPE,
    "attn_implementation": "eager" if MODE == "tiny_cpu" else "sdpa",
}
# Real 2B teaching runs default to QLoRA. CPU fixtures use ordinary LoRA.
USE_QLORA = MODE == "local_pretrained" and LESSON not in {"00a", "00b", "04"}
if USE_QLORA:
    from transformers import BitsAndBytesConfig

    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    load_kwargs["device_map"] = {"": torch.cuda.current_device()}
model = AutoModelForImageTextToText.from_pretrained(MODEL_PATH, **load_kwargs)
if not USE_QLORA:
    model.to(DEVICE)
model.config.use_cache = False
print(type(model).__name__, parameter_report(model))

## From local Q&A to train/validation/test records

A dataset row is not yet a tensor. Preserve the original group identity so examples from one conversation stay together. These tiny held-out splits demonstrate plumbing; use representative, larger splits in real experiments.


In [ ]:
# Convert local Q&A rows to canonical conversations; preserve provenance.
QA_FILE = Path(os.environ.get("FTLAB_QA_FILE", str(DATA_ROOT / "qa.csv")))
if QA_FILE.suffix.lower() == ".csv":
    with QA_FILE.open(encoding="utf-8", newline="") as handle:
        raw_rows = list(csv.DictReader(handle))
elif QA_FILE.suffix.lower() == ".jsonl":
    raw_rows = [
        json.loads(line)
        for line in QA_FILE.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
else:
    raise ValueError("This converter accepts CSV or JSONL Q&A files.")
for row in raw_rows:
    if (
        not row.get("group_id")
        or not row.get("question", "").strip()
        or not row.get("answer", "").strip()
    ):
        raise ValueError("Every Q&A needs a group_id, nonempty question, and nonempty answer.")
    row.setdefault("rejected", "")
records = [
    {
        "group_id": row["group_id"],
        "messages": [
            {"role": "user", "content": row["question"]},
            {"role": "assistant", "content": row["answer"]},
        ],
        "prompt": row["question"],
        "chosen": row["answer"],
        "rejected": row["rejected"],
    }
    for row in raw_rows
]


def split_records(rows, seed=SEED):
    # Deduplicate before splitting. Never split one document/conversation group.
    unique = {}
    for row in rows:
        key = json.dumps(row["messages"], sort_keys=True, ensure_ascii=False)
        unique.setdefault(key, row)
    groups = sorted({row["group_id"] for row in unique.values()})
    if len(groups) < 3:
        raise ValueError("Provide at least three independent document/conversation groups.")
    random.Random(seed).shuffle(groups)
    validation_groups, test_groups = set(groups[:1]), set(groups[1:2])
    splits = {"train": [], "validation": [], "test": []}
    for row in unique.values():
        split = (
            "validation"
            if row["group_id"] in validation_groups
            else "test"
            if row["group_id"] in test_groups
            else "train"
        )
        splits[split].append(row)
    return splits


splits = split_records(records)
for split, rows in splits.items():
    with (OUTPUT / f"{split}.jsonl").open("w", encoding="utf-8") as handle:
        for row in rows:
            canonical = {"group_id": row["group_id"], "messages": row["messages"]}
            handle.write(json.dumps(canonical, ensure_ascii=False) + "\n")
print({split: len(rows) for split, rows in splits.items()})
print("Raw:", raw_rows[0])
print("Canonical:", records[0])

## Chat rendering, tokenization and loss masking

The chat template supplies role delimiters. Attention masks describe real positions versus padding; labels choose prediction targets. A user token can be visible to attention while its label is `-100`. Assistant EOS should remain supervised even if its ID equals the padding ID.


In [ ]:
def render(messages, generation=False):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=generation,
        enable_thinking=False,
    )


def encode_conversation(messages):
    # Template-provided generation masks are preferred. They include assistant EOS.
    template = tokenizer.chat_template or ""
    if "generation" in template:
        encoded = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            return_dict=True,
            return_assistant_tokens_mask=True,
            enable_thinking=False,
        )
        ids = encoded["input_ids"]
        supervised = encoded["assistant_masks"]
    else:
        # For templates without generation annotations, verify prefix alignment.
        # Do not guess a token count by separately tokenizing the answer.
        ids = tokenizer(render(messages), add_special_tokens=False)["input_ids"]
        supervised = [0] * len(ids)
        for index, message in enumerate(messages):
            if message["role"] != "assistant":
                continue
            prefix = tokenizer(render(messages[:index], generation=True), add_special_tokens=False)[
                "input_ids"
            ]
            completed = tokenizer(render(messages[: index + 1]), add_special_tokens=False)[
                "input_ids"
            ]
            if ids[: len(prefix)] != prefix or ids[: len(completed)] != completed:
                raise ValueError(
                    "Template is not prefix-stable; use a training template with generation tags."
                )
            supervised[len(prefix) : len(completed)] = [1] * (len(completed) - len(prefix))
    if not any(supervised[1:]):
        raise ValueError("No assistant target tokens remain.")
    return {
        "input_ids": ids,
        "labels": [t if keep else -100 for t, keep in zip(ids, supervised, strict=False)],
    }


def collate_text(rows):
    items = [encode_conversation(row["messages"]) for row in rows]
    encoded = tokenizer.pad(
        [{"input_ids": item["input_ids"]} for item in items],
        padding=True,
        return_tensors="pt",
    )
    # Padding labels are independent of the pad token ID (pad may equal EOS).
    labels = torch.full_like(encoded["input_ids"], -100)
    for index, item in enumerate(items):
        labels[index, : len(item["labels"])] = torch.tensor(item["labels"])
    encoded["labels"] = labels
    return dict(encoded)


batch = collate_text(splits["train"][:2])
print(render(splits["train"][0]["messages"]))
print({name: tuple(value.shape) for name, value in batch.items()})
display(token_table(tokenizer, batch))

## Create or load local image annotations

Each image placeholder must correspond to one actual image. The default solid-color PNGs are generated instructional fixtures. Set `FTLAB_IMAGE_DATA` to your own JSONL file with `image` and `messages`, and `FTLAB_IMAGE_ROOT` for relative image paths. This lesson supervises the final assistant answer; reject ambiguous multi-answer samples rather than silently supervising the wrong span.


In [ ]:
from PIL import Image

image_file = os.environ.get("FTLAB_IMAGE_DATA")
if image_file:
    annotation_file = Path(image_file).expanduser().resolve()
    image_root = Path(os.environ.get("FTLAB_IMAGE_ROOT", str(annotation_file.parent)))
    image_rows = [
        json.loads(line) for line in annotation_file.read_text(encoding="utf-8").splitlines()
    ]
else:
    image_root = OUTPUT / "images"
    image_root.mkdir(exist_ok=True)
    image_rows = []
    for index, color in enumerate(["red", "blue", "green", "red", "blue", "green", "red"]):
        filename = f"{color}-{index}.png"
        fixture_image = Image.new("RGB", (32, 32), color)
        fixture_image.putpixel((index, 0), (255, 255, 255))
        fixture_image.save(image_root / filename)
        image_rows.append(
            {
                "image": filename,
                "messages": [
                    {
                        "role": "user",
                        "content": [
                            {"type": "image"},
                            {"type": "text", "text": "What is the color of image ?"},
                        ],
                    },
                    {"role": "assistant", "content": color},
                ],
            }
        )
if len(image_rows) < 3:
    raise ValueError("Supply at least three independent image examples.")
import hashlib

image_groups, image_owner, deduplicated = {}, {}, {}
for row in image_rows:
    with Image.open(image_root / row["image"]) as opened:
        digest = hashlib.sha256(opened.convert("RGB").tobytes()).hexdigest()
    group = row.get("group_id", digest)
    if digest in image_owner and image_owner[digest] != group:
        raise ValueError("Identical images have conflicting group IDs; reconcile provenance first.")
    image_owner[digest] = group
    deduplicated.setdefault((digest, json.dumps(row["messages"], sort_keys=True)), (group, row))
for group, row in deduplicated.values():
    image_groups.setdefault(group, []).append(row)
group_names = sorted(image_groups)
random.Random(SEED).shuffle(group_names)
if len(group_names) < 3:
    raise ValueError("Need three independent image/document groups after deduplication.")
splits = {
    "validation": image_groups[group_names[0]],
    "test": image_groups[group_names[1]],
    "train": [r for group in group_names[2:] for r in image_groups[group]],
}
print({name: len(rows) for name, rows in splits.items()})


def open_image(row):
    path = (image_root / row["image"]).resolve()
    with Image.open(path) as image:
        return image.convert("RGB")


display(open_image(splits["train"][0]))

## Inspect the processor output and construct a safe loss mask

`pixel_values` contains flattened image patches, not text token IDs. `image_grid_thw` describes their temporal/spatial grid. The processor expands one placeholder into the number of visual tokens required by that grid. We encode full conversations and prompt prefixes with the same images, then verify alignment. There is no token truncation; reject over-budget examples before the forward pass.


In [ ]:
MAX_IMAGE_SEQUENCE = 512 if MODE == "tiny_cpu" else 2048


def collate_text(rows):
    images = [open_image(row) for row in rows]
    full_texts, prompt_texts = [], []
    for row in rows:
        messages = row["messages"]
        if messages[-1]["role"] != "assistant":
            raise ValueError("The final message must be an assistant answer.")
        if sum(m["role"] == "assistant" for m in messages) != 1:
            raise ValueError("This lesson expects one assistant answer per image example.")
        full_texts.append(
            processor.apply_chat_template(
                messages,
                tokenize=False,
                enable_thinking=False,
            )
        )
        prompt_texts.append(
            processor.apply_chat_template(
                messages[:-1],
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        )
    encoded = processor(text=full_texts, images=images, padding=True, return_tensors="pt")
    if encoded["input_ids"].shape[1] > MAX_IMAGE_SEQUENCE:
        raise ValueError(
            "Image sequence exceeds the teaching budget; resize/filter, do not truncate."
        )
    labels = encoded["input_ids"].clone()
    labels[encoded["attention_mask"] == 0] = -100
    for index, (text, image) in enumerate(zip(prompt_texts, images, strict=False)):
        prefix = processor(text=[text], images=[image], return_tensors="pt")["input_ids"][0]
        if not torch.equal(encoded["input_ids"][index, : len(prefix)], prefix):
            raise ValueError("Image chat template is not prefix-stable.")
        labels[index, : len(prefix)] = -100
    for token_id in [
        model.config.image_token_id,
        model.config.video_token_id,
        model.config.vision_start_token_id,
        model.config.vision_end_token_id,
    ]:
        labels[encoded["input_ids"] == token_id] = -100
    if not (labels[:, 1:] != -100).any():
        raise ValueError("No answer tokens remain.")
    encoded["labels"] = labels
    return dict(encoded)


image_batch = collate_text(splits["train"][:2])
print({key: tuple(value.shape) for key, value in image_batch.items()})
display(token_table(tokenizer, image_batch))
assert (
    image_batch["labels"][image_batch["input_ids"] == model.config.image_token_id] == -100
).all()
# A deliberate short-budget probe must fail rather than cutting visual placeholders.
saved_budget = MAX_IMAGE_SEQUENCE
MAX_IMAGE_SEQUENCE = 1
try:
    collate_text(splits["train"][:1])
except ValueError as error:
    print("Expected truncation guard:", error)
else:
    raise AssertionError("Visual token truncation was not prevented.")
finally:
    MAX_IMAGE_SEQUENCE = saved_budget

## Train language adapters and the merger; freeze the vision encoder

The merger projects vision features into language embeddings. Saving it with the adapter preserves its updates. Inspect actual module names instead of assuming that all models share a projection name.


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# This is the same choice represented by tuning.strategy in a framework YAML.
TUNING = "lora"
if USE_QLORA and TUNING != "lora":
    raise ValueError("Full/selective tuning requires reloading with USE_QLORA=False.")
if USE_QLORA:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.enable_input_require_grads()
if TUNING == "lora":
    model = get_peft_model(
        model,
        LoraConfig(
            r=4 if MODE == "tiny_cpu" else 16,
            lora_alpha=8 if MODE == "tiny_cpu" else 32,
            target_modules="all-linear",
            exclude_modules=r".*(?:visual|vision).*",
            task_type="CAUSAL_LM",
            lora_dropout=0.0,
            modules_to_save=[
                name for name, _ in model.named_modules() if name.endswith("visual.merger")
            ],
        ),
    )
elif TUNING == "selective":
    for name, parameter in model.named_parameters():
        parameter.requires_grad = ("norm" in name or "lm_head" in name) and "visual" not in name
elif TUNING != "full":
    raise ValueError(TUNING)
trainable = [p for p in model.parameters() if p.requires_grad]
print(parameter_report(model))
# A small parameter sample avoids copying a 2B/4B model just to audit updates.
before = {name: p.detach().flatten()[:32].cpu().clone() for name, p in model.named_parameters()}
frozen_names = {name for name, p in model.named_parameters() if not p.requires_grad}

## Baseline and image-conditioned generation

Generation gets the image and the user message, never the reference answer. Validation loss supervises only the held-out assistant answer. Keep image resizing/patch geometry unchanged between training and inference.


In [ ]:
def to_device(batch):
    return {key: value.to(DEVICE) for key, value in batch.items()}


def precision_context():
    return torch.autocast("cuda", dtype=torch.bfloat16) if DEVICE.type == "cuda" else nullcontext()


def validation_loss(current_model, rows, collator=collate_text):
    current_model.eval()
    weighted_loss, target_count = 0.0, 0
    with torch.no_grad(), precision_context():
        for row in rows:
            encoded = to_device(collator([row]))
            count = int((encoded["labels"][:, 1:] != -100).sum())
            loss = current_model(**encoded, use_cache=False).loss
            weighted_loss += float(loss) * count
            target_count += count
    return weighted_loss / max(target_count, 1)


def generate_answer(current_model, prompt=None):
    row = splits["validation"][0]
    text = processor.apply_chat_template(
        row["messages"][:-1],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = processor(text=[text], images=[open_image(row)], return_tensors="pt")
    current_model.eval()
    with torch.no_grad(), precision_context():
        tokens = current_model.generate(
            **to_device(dict(inputs)),
            max_new_tokens=4 if MODE == "tiny_cpu" else 32,
            do_sample=False,
            use_cache=True,
        )
    return tokens[:, inputs["input_ids"].shape[1] :].cpu()


baseline_loss = validation_loss(model, splits["validation"])
baseline_answer = generate_answer(model)
print(
    {
        "baseline_image_loss": baseline_loss,
        "baseline_image_answer": tokenizer.decode(baseline_answer[0], skip_special_tokens=True),
    }
)

## Explicit multimodal training loop

The optimizer mechanics remain the same as text SFT. The difference is in processor outputs, visual token handling and the parameter selection.


In [ ]:
# The loss is averaged over valid target tokens, not padded positions.
train_loader = DataLoader(splits["train"][:5], batch_size=2, shuffle=False, collate_fn=collate_text)
ACCUMULATION = 2
EPOCHS = 1 if MODE == "tiny_cpu" else 2
optimizer = torch.optim.AdamW(trainable, lr=2e-3 if MODE == "tiny_cpu" else 2e-4)
updates_per_epoch = math.ceil(len(train_loader) / ACCUMULATION)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lambda step: max(0.1, 1 - step / (EPOCHS * updates_per_epoch)),
)
history = []
optimizer.zero_grad(set_to_none=True)
for _epoch in range(EPOCHS):
    batches = list(train_loader)  # Tiny lesson only; stream windows for large corpora.
    for window_start in range(0, len(batches), ACCUMULATION):
        window = batches[window_start : window_start + ACCUMULATION]
        counts = [int((item["labels"][:, 1:] != -100).sum()) for item in window]
        total_targets = sum(counts)
        if total_targets == 0:
            raise ValueError("This accumulation window has no supervised targets.")
        model.train()
        for batch, count in zip(window, counts, strict=False):
            with precision_context():
                outputs = model(**to_device(batch), use_cache=False)
                # Correct even for the final short window and unequal token lengths.
                loss = outputs.loss * count / total_targets
            assert torch.isfinite(loss)
            loss.backward()
            history.append(float(outputs.loss.detach()))
        assert all(torch.isfinite(p.grad).all() for p in trainable if p.grad is not None)
        grad_norm = torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        print({"update": scheduler.last_epoch, "loss": history[-1], "grad_norm": float(grad_norm)})
assert scheduler.last_epoch == EPOCHS * updates_per_epoch

## Inspect updates and held-out behavior

A finite loss and a changed adapter prove an update occurred, not that a model became useful. Inspect validation loss and example generations together. Keep the test set out of hyperparameter selection.


In [ ]:
changed = []
for name, parameter in model.named_parameters():
    same = torch.equal(before[name], parameter.detach().flatten()[:32].cpu())
    if name in frozen_names:
        assert same, f"Frozen parameter changed: {name}"
    elif not same:
        changed.append(name)
assert changed, "No trainable parameter sample changed."
after_loss = validation_loss(model, splits["validation"])
after_answer = generate_answer(model)
test_loss = validation_loss(model, splits["test"])
print(
    {
        "baseline_loss": baseline_loss,
        "after_loss": after_loss,
        "test_loss": test_loss,
        "changed_parameter_samples": changed[:5],
    }
)
print("Before:", tokenizer.decode(baseline_answer[0], skip_special_tokens=True))
print("After: ", tokenizer.decode(after_answer[0], skip_special_tokens=True))
# Do not assert that generalization improves after one synthetic update.
assert math.isfinite(after_loss) and math.isfinite(test_loss)

## Save, reload, and verify

`save_pretrained()` saves inference artifacts; it does not save the optimizer or training position. A PEFT artifact needs its original base. This cell creates a separate model object and compares generated token IDs. Lesson 08 covers exact resume and merging.


In [ ]:
from peft import PeftModel

artifact = OUTPUT / "final"
model.save_pretrained(artifact, safe_serialization=True)
processor.save_pretrained(artifact)
expected_tokens = generate_answer(model)
reloaded_processor = AutoProcessor.from_pretrained(artifact, local_files_only=True)
assert reloaded_processor.tokenizer.get_vocab() == tokenizer.get_vocab()
assert reloaded_processor.chat_template == processor.chat_template
processor = reloaded_processor
tokenizer = processor.tokenizer
# Reload independently, rather than reusing the trained Python object.
if (artifact / "adapter_config.json").exists():
    reload_base = AutoModelForImageTextToText.from_pretrained(MODEL_PATH, **load_kwargs)
    if not USE_QLORA:
        reload_base.to(DEVICE)
    reloaded = PeftModel.from_pretrained(reload_base, artifact, local_files_only=True)
else:
    reloaded = AutoModelForImageTextToText.from_pretrained(artifact, **load_kwargs)
    if not USE_QLORA:
        reloaded.to(DEVICE)
actual_tokens = generate_answer(reloaded)
assert torch.equal(expected_tokens, actual_tokens), "Greedy outputs changed after reload."
report = {
    "mode": MODE,
    "base_checkpoint": str(MODEL_PATH),
    "artifact": str(artifact),
    "baseline_validation_loss": baseline_loss,
    "validation_loss": after_loss,
    "test_loss": test_loss,
    "reload_tokens_equal": True,
}
(OUTPUT / "lesson_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(report)
del reloaded
if "reload_base" in globals():
    del reload_base
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Interpretation, common failures, and exercises

- If loss is NaN, inspect the number of supervised targets, precision and learning rate before adding steps.
- If every label is `-100`, repair the template/mask; an empty objective cannot teach anything.
- If frozen parameters change, inspect the trainable parameter report and optimizer parameter list.
- If tiny generations look meaningless, that is expected from random initialization and a tiny vocabulary.

**Exercises:** (1) Print which token predicts the first answer token. (2) Compare full/selective/LoRA parameter counts. (3) Change one training answer, rerun from the same seed, and inspect held-out loss. (4) Explain why saving an adapter is not enough to resume AdamW.

**Expected result:** finite objective values, some expected trainable weights changed, frozen weights unchanged, and identical greedy tokens after reload. Record actual values in `lesson_report.json`; no fixed quality threshold is asserted.
